# GraphKV Position Ablation

这个 Notebook 专门回答一个问题：

> **Graph-KV 的输出差异，究竟来自 Graph-KV 的结构消息传递，还是来自 source/target positional encoding 的实现方式？**

本实验在**同一个本地 Llama-3.1-8B-Block-FT 模型、同一个 GPU、同一组 RAG 输入、同一个 query**上比较三条路径：

### A. Sequential baseline
```text
Prefix + Context1 + Context2 + ... + Query
```

### B. Official Graph-KV reproduction
按照当前 `Graph-COM/GraphKV` `pcw.py` 的 cache/RoPE 流程复现：
```text
独立 context KV
→ source KV 归一到 position 0
→ target 读取 source KV
→ target KV 再归一
→ Prefix + Source + Target
→ 最终对整个 cache 重新施加连续 position 0...N-1
→ Query
```

### C. Paper-faithful Graph-KV
按照论文 v2 第 3.1 节的 positional design：
```text
Prefix      : [0, P)
Source      : [P, P+L)          （所有 source 共享）
Target      : [P+L, P+2L)       （所有 target 共享）
Query       : [P+2L, ...)
```

论文明确描述 source/target 使用共享 positional ranges；当前官方 `pcw.py` 则在最终 concat 后再次对整个 KV cache 使用连续 position。fileciteturn0file0L279-L301
官方仓库当前 `pcw.py` 的相应 cache/RoPE 流程见其 `gapemp()` 实现。citeturn505647view0

---

## 重要

这个 Notebook 不比较最终生成字符串，而是直接比较 **next-token logits**：

- Top-10 token
- EOS / 正确答案 token 的 logit
- probability
- Pairwise KL divergence
- 三种方法的 position layout
- Final physical KV length

这样可以直接定位 positional implementation 对模型分布的影响。


In [3]:
from pathlib import Path

# ============================================================
# 1. 实验配置：只需要修改这里
# ============================================================

GPU_ID = 3
MODEL_INDEX = 0

MODELS_ROOT = Path("/home/lyh/hcy_work/GraphKV/models")

# 最大生成长度不在本消融中使用；这里只测 query 后的第一步 logits。
TOP_K = 10

print("GPU_ID:", GPU_ID)
print("MODEL_INDEX:", MODEL_INDEX)
print("MODELS_ROOT:", MODELS_ROOT)


GPU_ID: 3
MODEL_INDEX: 0
MODELS_ROOT: /home/lyh/hcy_work/GraphKV/models


In [4]:
# ============================================================
# 2. 环境与 CUDA 检查
# ============================================================

import sys
import os
import gc
import importlib
import math
import traceback
from pathlib import Path

import torch
import torch.nn.functional as F

assert torch.cuda.is_available(), "CUDA is required."
assert 0 <= GPU_ID < torch.cuda.device_count()

print("torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    free_b, total_b = torch.cuda.mem_get_info(i)
    print(
        f"GPU {i}: {torch.cuda.get_device_name(i)} | "
        f"free={free_b/2**30:.2f} GB / {total_b/2**30:.2f} GB"
    )

print()
print("Selected physical GPU:", GPU_ID)


torch: 2.5.1+cu121
CUDA: 12.1
GPU count: 4
GPU 0: NVIDIA GeForce RTX 4090 | free=21.67 GB / 23.64 GB
GPU 1: NVIDIA GeForce RTX 4090 | free=21.00 GB / 23.64 GB
GPU 2: NVIDIA GeForce RTX 4090 | free=15.58 GB / 23.64 GB
GPU 3: NVIDIA GeForce RTX 4090 | free=23.26 GB / 23.64 GB

Selected physical GPU: 3


## 3. 清理 Jupyter 旧模块

确保 Notebook 使用当前目录里的重构代码，而不是旧的 Python module cache。


In [5]:
for name in [
    "graph_kv_cache",
    "graph_kv_adapter",
    "pcw",
    "pcw_parallel",
]:
    sys.modules.pop(name, None)

importlib.invalidate_caches()

import graph_kv_cache
import graph_kv_adapter
import pcw
import pcw_parallel

from graph_kv_cache import GraphKVCache
from graph_kv_adapter import _encode_independent_chunk, _tokenize_one
from pcw import (
    apply_pkv_rerotary_position_embeddings,
    apply_pkv_rotary_position_embeddings,
)

print("graph_kv_cache :", graph_kv_cache.__file__)
print("graph_kv_adapter:", graph_kv_adapter.__file__)
print("pcw            :", pcw.__file__)
print("pcw_parallel   :", pcw_parallel.__file__)


/home/lyh/miniconda3/envs/graphkv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


graph_kv_cache : /home/lyh/hcy_work/GraphKV/graph_kv_cache.py
graph_kv_adapter: /home/lyh/hcy_work/GraphKV/graph_kv_adapter.py
pcw            : /home/lyh/hcy_work/GraphKV/pcw.py
pcw_parallel   : /home/lyh/hcy_work/GraphKV/pcw_parallel.py


In [6]:
# ============================================================
# 4. Transformers / FlashAttention
# ============================================================

import transformers
import flash_attn

print("transformers:", transformers.__version__)
print("flash_attn:", flash_attn.__version__)

assert transformers.__version__ == "4.50.0"


transformers: 4.50.0
flash_attn: 2.7.4.post1


## 5. 选择本地模型

默认：
```text
/home/lyh/hcy_work/GraphKV/models
```


In [7]:
model_dirs = sorted(
    p for p in MODELS_ROOT.iterdir()
    if p.is_dir() and (p / "config.json").exists()
)

for i, p in enumerate(model_dirs):
    print(i, p.name)

selected_model_dir = model_dirs[MODEL_INDEX]
print("\nSelected:", selected_model_dir)


0 Tulu3-Block-FT
1 Tulu3-RAG
2 Tulu3-SFT

Selected: /home/lyh/hcy_work/GraphKV/models/Tulu3-Block-FT


## 6. 加载单卡模型

本实验**不使用 `device_map="auto"`**。

模型全部放在 `GPU_ID`：
```python
device_map={"": GPU_ID}
```


In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(
    str(selected_model_dir),
    use_fast=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    str(selected_model_dir),
    torch_dtype=torch.bfloat16,
    device_map={"": GPU_ID},
    attn_implementation="flash_attention_2",
    low_cpu_mem_usage=True,
)
model.eval()

parameter_devices = {str(p.device) for p in model.parameters()}

print("Model class:", type(model).__name__)
print("hf_device_map:", getattr(model, "hf_device_map", None))
print("parameter devices:", parameter_devices)

assert parameter_devices == {f"cuda:{GPU_ID}"}

# Single-GPU rotary embedding
emb = getattr(getattr(model, "model", None), "rotary_emb", None)
print("rotary:", type(emb).__name__ if emb is not None else None)


Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.05it/s]


Model class: LlamaForCausalLM
hf_device_map: {'': 3}
parameter devices: {'cuda:3'}
rotary: LlamaRotaryEmbedding


## 7. 固定实验输入

这里使用与前面调试一致的简单 RAG 场景，但改为更接近论文 A.2.1 的 RAG prompt 形式。

论文的 RAG prompt 分成 Prefix / Text Chunks / Question；Graph-KV、Sequential、Parallel 方法使用相同 prompt 内容。fileciteturn3file1

为了让三种方法只在 KV/position 机制上不同，不改变 query 字符串。


In [9]:
prefix = (
    "You are an intelligent AI assistant. "
    "Please answer questions based on the user's instructions. "
    "Below are some reference documents that may help you in answering the user's question."
)

query = (
    "Please write a high-quality answer for the given question using only "
    "the provided search documents (some of which might be irrelevant). "
    "Question: What is the capital city of France?"
)

contexts = [
    "-Title: Geography of France.\n France is a country in Europe.",
    "-Title: Paris.\n Paris is the capital city of France.",
    "-Title: Germany.\n Berlin is the capital city of Germany.",
    "-Title: Eiffel Tower.\n The Eiffel Tower is located in Paris.",
]

print("Prefix:")
print(prefix)
print("\nQuery:")
print(query)
print("\nContexts:")
for i, text in enumerate(contexts):
    print(f"[{i}] {text}")


Prefix:
You are an intelligent AI assistant. Please answer questions based on the user's instructions. Below are some reference documents that may help you in answering the user's question.

Query:
Please write a high-quality answer for the given question using only the provided search documents (some of which might be irrelevant). Question: What is the capital city of France?

Contexts:
[0] -Title: Geography of France.
 France is a country in Europe.
[1] -Title: Paris.
 Paris is the capital city of France.
[2] -Title: Germany.
 Berlin is the capital city of Germany.
[3] -Title: Eiffel Tower.
 The Eiffel Tower is located in Paris.


In [10]:
# Token lengths used by all three methods
prefix_ids = _tokenize_one(tokenizer, prefix, torch.device(f"cuda:{GPU_ID}"))
query_ids = _tokenize_one(tokenizer, query, torch.device(f"cuda:{GPU_ID}"))

context_ids = [
    _tokenize_one(tokenizer, text, torch.device(f"cuda:{GPU_ID}"))
    for text in contexts
]

print("prefix length:", prefix_ids.shape[-1])
print("query length :", query_ids.shape[-1])
print("context lens :", [int(x.shape[-1]) for x in context_ids])
print("source span L:", max(int(x.shape[-1]) for x in context_ids))


prefix length: 33
query length : 34
context lens : [14, 13, 13, 18]
source span L: 18


# A. Sequential baseline

这里不使用 KV graph。

直接把：
```text
Prefix + all contexts + Query
```
作为普通 causal sequence。

然后只取最后一个 query token 的 next-token logits。


In [11]:
@torch.inference_mode()
def sequential_next_logits(model, tokenizer, prefix, contexts, query):
    prompt = (
        prefix
        + "\n\n"
        + "\n".join(contexts)
        + "\n\n"
        + query
    )

    ids = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
        truncation=False,
    ).input_ids.to(model.device)

    out = model(
        ids,
        use_cache=True,
    )

    return {
        "logits": out.logits[:, -1, :].detach().float().cpu(),
        "input_ids": ids.detach().cpu(),
        "prompt": prompt,
    }

seq_result = sequential_next_logits(
    model,
    tokenizer,
    prefix,
    contexts,
    query,
)

seq_logits = seq_result["logits"]
print("Sequential input length:", seq_result["input_ids"].shape[-1])


Sequential input length: 125


# B. Official Graph-KV reproduction

下面尽量保持当前官方 `pcw.py::gapemp()` 的 cache construction / RoPE 逻辑：

1. independently encode contexts
2. remove RoPE to position 0
3. flatten source chunks
4. apply continuous positions to source KV
5. let target chunks attend those source KVs
6. extract target KV
7. remove target RoPE
8. encode Prefix
9. concatenate Prefix + Source + Target
10. **apply continuous RoPE across the entire final cache**
11. Query forward

这一步专门用于复现“官方仓库实际行为”，而不是声称它等价于论文文字中的 shared source/target PE。


In [12]:
def clone_dynamic_cache(cache):
    new_cache = type(cache)()
    new_cache.key_cache = [x.clone().detach() for x in cache.key_cache]
    new_cache.value_cache = [x.clone().detach() for x in cache.value_cache]
    return new_cache


def flatten_pkv_official(pkv, mask):
    for layer_id in range(len(pkv.key_cache)):
        pkv.key_cache[layer_id] = (
            pkv.key_cache[layer_id]
            .transpose(1, 2)
            .flatten(0, 1)[mask]
            .unsqueeze(0)
            .transpose(1, 2)
        )
        pkv.value_cache[layer_id] = (
            pkv.value_cache[layer_id]
            .transpose(1, 2)
            .flatten(0, 1)[mask]
            .unsqueeze(0)
            .transpose(1, 2)
        )
    return pkv


def stack_pkv_official(pkv, bsz):
    for layer_id in range(len(pkv.key_cache)):
        pkv.key_cache[layer_id] = pkv.key_cache[layer_id].repeat(
            bsz, 1, 1, 1
        )
        pkv.value_cache[layer_id] = pkv.value_cache[layer_id].repeat(
            bsz, 1, 1, 1
        )
    return pkv


def concat_pkv_official(pkv1, pkv2):
    for layer_id in range(len(pkv2.key_cache)):
        pkv2.key_cache[layer_id] = torch.cat(
            (pkv1.key_cache[layer_id], pkv2.key_cache[layer_id]),
            dim=2,
        )
        pkv2.value_cache[layer_id] = torch.cat(
            (pkv1.value_cache[layer_id], pkv2.value_cache[layer_id]),
            dim=2,
        )
    return pkv2


def cut_pkv_official(pkv, positions):
    for layer_id in range(len(pkv.key_cache)):
        pkv.key_cache[layer_id] = pkv.key_cache[layer_id][:, :, positions, :]
        pkv.value_cache[layer_id] = pkv.value_cache[layer_id][:, :, positions, :]
    return pkv


@torch.inference_mode()
def official_graphkv_next_logits(
    model,
    tokenizer,
    emb,
    prefix,
    contexts,
    query,
):
    device = model.device

    prefix_ids = _tokenize_one(tokenizer, prefix, device)
    query_ids = _tokenize_one(tokenizer, query, device)

    # Match the official implementation's batched context prefill.
    context_batch = tokenizer(
        contexts,
        return_tensors="pt",
        truncation=False,
        padding=True,
        add_special_tokens=False,
    ).input_ids.to(device)

    context_mask = (context_batch != tokenizer.pad_token_id).reshape(-1)
    bsz = context_batch.shape[0]
    len_flatten_contexts = int(context_mask.sum().item())

    # Initial context KV
    context_outputs = model(
        context_batch,
        use_cache=True,
    )
    context_pkv = context_outputs.past_key_values

    # Official: normalize source K to local position 0
    context_pkv = apply_pkv_rerotary_position_embeddings(
        context_pkv,
        emb,
    )
    context_pkv = flatten_pkv_official(
        context_pkv,
        context_mask,
    )

    # Clone source KV at local position 0 for final merge.
    context_pkv_clone = clone_dynamic_cache(context_pkv)

    # Official: source chunks occupy a continuous range.
    context_pkv = apply_pkv_rotary_position_embeddings(
        context_pkv,
        emb,
    )
    context_pkv = stack_pkv_official(context_pkv, bsz)

    # Target chunks attend only to the source KV.
    context2_outputs = model(
        context_batch,
        past_key_values=context_pkv,
        use_cache=True,
    )
    context_pkv = context2_outputs.past_key_values

    position_of_context2 = (
        torch.arange(
            start=0,
            end=context_batch.shape[-1],
            dtype=torch.long,
            device=device,
        )
        + len_flatten_contexts
    )

    # Keep target portion and normalize it back to local position 0.
    context_pkv = cut_pkv_official(
        context_pkv,
        position_of_context2,
    )
    context_pkv = apply_pkv_rerotary_position_embeddings(
        context_pkv,
        emb,
        position_of_context2,
    )
    context_pkv = flatten_pkv_official(
        context_pkv,
        context_mask,
    )

    # Final source + target merge.
    context_pkv = concat_pkv_official(
        context_pkv_clone,
        context_pkv,
    )

    # Prefix KV.
    prefix_outputs = model(
        prefix_ids,
        past_key_values=None,
        use_cache=True,
    )
    prefix_pkv = prefix_outputs.past_key_values
    prefix_pkv = apply_pkv_rerotary_position_embeddings(
        prefix_pkv,
        emb,
    )

    # Prefix + source + target.
    context_pkv = concat_pkv_official(
        prefix_pkv,
        context_pkv,
    )

    # IMPORTANT: official repository re-applies continuous RoPE to
    # the complete final KV cache.
    context_pkv = apply_pkv_rotary_position_embeddings(
        context_pkv,
        emb,
    )

    # Query: official implementation relies on implicit continuation of
    # physical cache positions.
    outputs = model(
        query_ids,
        past_key_values=context_pkv,
        use_cache=True,
    )

    return {
        "logits": outputs.logits[:, -1, :].detach().float().cpu(),
        "cache": context_pkv,
        "query_ids": query_ids.detach().cpu(),
        "source_physical_len": len_flatten_contexts,
        "target_physical_len": len_flatten_contexts,
    }


official_result = official_graphkv_next_logits(
    model,
    tokenizer,
    emb,
    prefix,
    contexts,
    query,
)

official_logits = official_result["logits"]

print(
    "Official physical KV len:",
    official_result["cache"].key_cache[0].shape[-2],
)


Official physical KV len: 183


# C. Paper-faithful Graph-KV

这里使用重构后的 `GraphKVCache`。

关键区别：

```text
Prefix:
[0, P)

Source:
[P, P+L)        # all source nodes share the same logical range

Target:
[P+L, P+2L)    # all first-hop targets share the next range

Query:
[P+2L, ...)
```

物理 KV 可以继续按：
```text
Prefix + Source1 + Source2 + ... + Target1 + Target2 + ...
```
拼接。

但是 logical `position_ids` 与 physical `cache_position` 完全分离。


In [13]:
from graph_kv_adapter import _tokenize_one, _encode_independent_chunk

@torch.inference_mode()
def paper_faithful_next_logits(
    model,
    tokenizer,
    prefix,
    contexts,
    query,
):
    device = model.device

    prefix_ids = _tokenize_one(tokenizer, prefix, device)
    query_ids = _tokenize_one(tokenizer, query, device)

    # Prefix is independently encoded at logical positions [0, P).
    prefix_cache = _encode_independent_chunk(
        model,
        prefix_ids,
        position_start=0,
    )

    P = int(prefix_ids.shape[-1])

    gkv = GraphKVCache(
        source_position_start=P,
    )
    gkv.set_prefix_cache(prefix_cache)

    source_ids = []
    target_ids = []

    # 1) Source nodes: independently encoded at logical [P, P + len_i).
    for i, text in enumerate(contexts):
        node_id = f"source::{i}"
        ids = _tokenize_one(tokenizer, text, device)

        cache = _encode_independent_chunk(
            model,
            ids,
            position_start=P,
        )

        gkv.add_source(
            node_id,
            cache,
            assume_positioned=True,
        )
        source_ids.append(node_id)

    # 2) Every target reads all source nodes (Graph-KV-Full).
    for i, text in enumerate(contexts):
        target_id = f"target::{i}"
        ids = _tokenize_one(tokenizer, text, device)

        for sid in source_ids:
            gkv.add_edge(sid, target_id)

        gkv.propagate_target(
            model,
            target_id,
            ids,
            source_ids=source_ids,
            round_idx=1,
        )

        target_ids.append(target_id)

    gkv.validate()

    # 3) Query uses logical [P + L, P + 2L).
    query_cache, query_position_ids, query_cache_position = (
        gkv.prepare_query(
            query_ids,
            source_ids=source_ids,
            target_ids=target_ids,
        )
    )

    outputs = model(
        query_ids,
        past_key_values=query_cache,
        position_ids=query_position_ids,
        cache_position=query_cache_position,
        use_cache=True,
    )

    return {
        "logits": outputs.logits[:, -1, :].detach().float().cpu(),
        "gkv": gkv,
        "query_position_ids": query_position_ids.detach().cpu(),
        "query_cache_position": query_cache_position.detach().cpu(),
        "query_cache": query_cache,
        "source_ids": source_ids,
        "target_ids": target_ids,
    }


paper_result = paper_faithful_next_logits(
    model,
    tokenizer,
    prefix,
    contexts,
    query,
)

paper_logits = paper_result["logits"]

print("Paper-faithful cache length:",
      paper_result["query_cache"].key_cache[0].shape[-2])
print("Query logical positions:",
      paper_result["query_position_ids"].tolist())
print("Query physical positions:",
      paper_result["query_cache_position"].tolist())


Paper-faithful cache length: 183
Query logical positions: [[69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102]]
Query physical positions: [149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182]


# 12. Position layout audit

这一节是本实验最重要的结果之一。


In [14]:
def print_position_audit(gkv, prefix_len, label):
    print("=" * 80)
    print(label)
    print("=" * 80)

    print("prefix logical:", f"[0, {prefix_len})")
    print("source logical ranges:")

    for node_id, meta in gkv.meta.items():
        if meta.kind == "source":
            print(
                f"  {node_id}: "
                f"[{meta.logical_start}, {meta.logical_end})"
            )

    print("target logical ranges:")
    for node_id, meta in gkv.meta.items():
        if meta.kind == "target":
            print(
                f"  {node_id}: "
                f"[{meta.logical_start}, {meta.logical_end})"
            )

    print(
        "query logical:",
        paper_result["query_position_ids"].tolist(),
    )

    print(
        "query physical:",
        paper_result["query_cache_position"].tolist(),
    )


print_position_audit(
    paper_result["gkv"],
    int(prefix_ids.shape[-1]),
    "C. Paper-faithful Graph-KV",
)

print()
print("A. Sequential:")
print(
    "  physical/logical input positions are simply "
    f"[0, {seq_result['input_ids'].shape[-1]})"
)

print()
print("B. Official Graph-KV:")
print(
    "  final cache is re-RoPE'd continuously over "
    f"[0, {official_result['cache'].key_cache[0].shape[-2]})"
)


C. Paper-faithful Graph-KV
prefix logical: [0, 33)
source logical ranges:
  source::0: [33, 47)
  source::1: [33, 46)
  source::2: [33, 46)
  source::3: [33, 51)
target logical ranges:
  target::0: [51, 65)
  target::1: [51, 64)
  target::2: [51, 64)
  target::3: [51, 69)
query logical: [[69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102]]
query physical: [149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182]

A. Sequential:
  physical/logical input positions are simply [0, 125)

B. Official Graph-KV:
  final cache is re-RoPE'd continuously over [0, 183)


# 13. Top-10 next-token comparison

In [15]:
def topk_table(logits, tokenizer, k=10):
    probs = torch.softmax(logits.float(), dim=-1)
    values, indices = torch.topk(probs, k=k, dim=-1)

    rows = []
    for rank, (p, idx) in enumerate(zip(values[0], indices[0]), 1):
        tid = int(idx.item())
        rows.append({
            "rank": rank,
            "token_id": tid,
            "probability": float(p.item()),
            "logit": float(logits[0, tid].item()),
            "text": tokenizer.decode(
                [tid],
                skip_special_tokens=False,
            ),
            "is_eos": (
                tokenizer.eos_token_id is not None
                and tid == int(tokenizer.eos_token_id)
            ),
        })
    return rows


for name, logits in [
    ("Sequential", seq_logits),
    ("Official Graph-KV", official_logits),
    ("Paper-faithful Graph-KV", paper_logits),
]:
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    for row in topk_table(logits, tokenizer, TOP_K):
        print(
            f"{row['rank']:2d}. "
            f"id={row['token_id']:6d} "
            f"p={row['probability']:.6f} "
            f"logit={row['logit']:.4f} "
            f"EOS={row['is_eos']} "
            f"text={row['text']!r}"
        )



Sequential
 1. id=128001 p=0.445733 logit=15.8750 EOS=True text='<|end_of_text|>'
 2. id=  2650 p=0.053235 logit=13.7500 EOS=False text=' How'
 3. id= 22559 p=0.050010 logit=13.6875 EOS=False text=' Answer'
 4. id=  4815 p=0.046980 logit=13.6250 EOS=False text=' \n\n'
 5. id=  3639 p=0.044134 logit=13.5625 EOS=False text=' What'
 6. id=   320 p=0.041460 logit=13.5000 EOS=False text=' ('
 7. id= 12366 p=0.022192 logit=12.8750 EOS=False text=' Paris'
 8. id=   720 p=0.019584 logit=12.7500 EOS=False text=' \n'
 9. id=  5321 p=0.016236 logit=12.5625 EOS=False text=' Please'
10. id= 15425 p=0.014328 logit=12.4375 EOS=False text='"""\n\n'

Official Graph-KV
 1. id=128001 p=0.323628 logit=15.6250 EOS=True text='<|end_of_text|>'
 2. id= 16533 p=0.285601 logit=15.5000 EOS=False text='Answer'
 3. id= 22559 p=0.063726 logit=14.0000 EOS=False text=' Answer'
 4. id=    32 p=0.023444 logit=13.0000 EOS=False text='A'
 5. id=  4815 p=0.019435 logit=12.8125 EOS=False text=' \n\n'
 6. id= 60704 p=0.015

# 14. 自动检查 EOS / 答案 token

这个 cell 不假设具体 tokenization，只查看：

- EOS probability
- `Paris` tokenization 的 probability
- three-way argmax


In [16]:
def token_sequence_probs(logits, token_ids):
    probs = torch.softmax(logits.float(), dim=-1)
    vals = []
    for tid in token_ids:
        vals.append(float(probs[0, int(tid)].item()))
    return vals


paris_ids = tokenizer(
    " Paris",
    add_special_tokens=False,
).input_ids

print("Tokenization of ' Paris':", paris_ids)

for name, logits in [
    ("Sequential", seq_logits),
    ("Official Graph-KV", official_logits),
    ("Paper-faithful Graph-KV", paper_logits),
]:
    eos_prob = (
        None
        if tokenizer.eos_token_id is None
        else float(
            torch.softmax(logits.float(), dim=-1)[
                0, tokenizer.eos_token_id
            ].item()
        )
    )

    paris_probs = token_sequence_probs(logits, paris_ids)

    argmax_id = int(logits.argmax(dim=-1)[0].item())
    argmax_text = tokenizer.decode(
        [argmax_id],
        skip_special_tokens=False,
    )

    print("\n", name)
    print("  EOS probability:", eos_prob)
    print("  ' Paris' token probabilities:", paris_probs)
    print("  argmax:", argmax_id, repr(argmax_text))


Tokenization of ' Paris': [12366]

 Sequential
  EOS probability: 0.4457329213619232
  ' Paris' token probabilities: [0.022191734984517097]
  argmax: 128001 '<|end_of_text|>'

 Official Graph-KV
  EOS probability: 0.32362788915634155
  ' Paris' token probabilities: [0.011788145639002323]
  argmax: 128001 '<|end_of_text|>'

 Paper-faithful Graph-KV
  EOS probability: 0.37384191155433655
  ' Paris' token probabilities: [0.012017128989100456]
  argmax: 128001 '<|end_of_text|>'


# 15. KL divergence

定义：
```text
KL(A || B)
```

表示 A 的 next-token distribution 视为 reference 时，B 偏离 A 的程度。

注意：
KL 不是对称距离，所以同时报告：
- Sequential || Official
- Official || Sequential
- Sequential || Paper
- Paper || Sequential
- Official || Paper
- Paper || Official


In [17]:
def kl_divergence(logits_a, logits_b):
    log_p = torch.log_softmax(logits_a.float(), dim=-1)
    q = torch.softmax(logits_b.float(), dim=-1)
    return float(F.kl_div(log_p, q, reduction="sum").item())


pairs = [
    ("Sequential || Official", seq_logits, official_logits),
    ("Official || Sequential", official_logits, seq_logits),
    ("Sequential || Paper", seq_logits, paper_logits),
    ("Paper || Sequential", paper_logits, seq_logits),
    ("Official || Paper", official_logits, paper_logits),
    ("Paper || Official", paper_logits, official_logits),
]

for name, a, b in pairs:
    print(f"{name:28s}: {kl_divergence(a, b):.8f}")


Sequential || Official      : 1.06284428
Official || Sequential      : 0.69374746
Sequential || Paper         : 0.85160130
Paper || Sequential         : 0.64471674
Official || Paper           : 0.02541555
Paper || Official           : 0.02562247


# 16. 最终结果摘要

重点观察：

1. Sequential 与 Official 是否接近？
2. Official 与 Paper-faithful 是否接近？
3. Paper-faithful 是否出现 EOS 概率显著上升？
4. Paper-faithful 的 source/target position 是否确实重叠共享？
5. 三者的 physical KV length 是否一致。

### 解释原则

- 如果 **Official ≈ Sequential**，但 **Paper-faithful 明显不同**：
  positional re-encoding 很可能是主要变量。
- 如果 **Official ≈ Paper-faithful**，但两者都和 Sequential 差异很大：
  主要差异更可能来自 Graph-KV 的 source→target message passing。
- 如果三个分布都非常不同：
  需要继续做 attention mask / KV content ablation。


In [18]:
summary = {
    "sequential_input_len": int(seq_result["input_ids"].shape[-1]),
    "official_final_cache_len": int(
        official_result["cache"].key_cache[0].shape[-2]
    ),
    "paper_final_cache_len": int(
        paper_result["query_cache"].key_cache[0].shape[-2]
    ),
    "paper_query_logical_start": int(
        paper_result["query_position_ids"][0, 0].item()
    ),
    "paper_query_physical_start": int(
        paper_result["query_cache_position"][0].item()
    ),
    "argmax": {
        "sequential": int(seq_logits.argmax(dim=-1)[0].item()),
        "official": int(official_logits.argmax(dim=-1)[0].item()),
        "paper": int(paper_logits.argmax(dim=-1)[0].item()),
    },
    "kl": {
        name: kl_divergence(a, b)
        for name, a, b in pairs
    },
}

import pprint
pprint.pp(summary)


{'sequential_input_len': 125,
 'official_final_cache_len': 183,
 'paper_final_cache_len': 183,
 'paper_query_logical_start': 69,
 'paper_query_physical_start': 149,
 'argmax': {'sequential': 128001, 'official': 128001, 'paper': 128001},
 'kl': {'Sequential || Official': 1.0628442764282227,
        'Official || Sequential': 0.6937474608421326,
        'Sequential || Paper': 0.8516013026237488,
        'Paper || Sequential': 0.644716739654541,
        'Official || Paper': 0.025415552780032158,
        'Paper || Official': 0.025622468441724777}}


## 17. 释放 GPU

实验结束后再运行。


In [19]:
del model
del tokenizer
gc.collect()
torch.cuda.empty_cache()
print("Model released.")


Model released.
